# pm_spark — Databricks import check

Run on a **Databricks** cluster (e.g. **DBR 14.3 LTS**, Spark 3.5) **after** the wheel is installed.

See the repository **README.md**, section **Databricks smoke test**, for upload paths and `%pip` examples.

## 1. Install the wheel (if not already a cluster library)

Uncomment **one** line and set the path to your `.whl`. Use **`--no-deps`** so the cluster keeps its bundled PySpark.

In [ ]:
# %pip install /dbfs/FileStore/libs/pm_spark-0.1.0-py3-none-any.whl --no-deps
# %pip install /Volumes/<catalog>/<schema>/<volume>/wheels/pm_spark-0.1.0-py3-none-any.whl --no-deps

If you used `%pip`, **restart Python** (Databricks: **Run** → **Restart Python**) or reattach the notebook, then run the cells below.

## 2. Package version

In [ ]:
import pm_spark

print("pm_spark", pm_spark.__version__)

## 3. Flat imports (`pm_spark` re-exports public functions)

In [ ]:
from pm_spark import activity_sequence_number, throughput_time

print(activity_sequence_number.__name__, throughput_time.__name__)

## 4. Submodule style (same functions, explicit module path)

In [ ]:
from pm_spark.eventlog import preparation as prep

print(prep.activity_sequence_number.__name__)

## 5. One transform on the cluster `spark` session

In [ ]:
from datetime import datetime

from pyspark.sql import Row

from pm_spark import activity_sequence_number

test_df = spark.createDataFrame(
    [
        Row(case_id="CHK1", activity="A", event_time=datetime(2024, 1, 1, 10, 0)),
        Row(case_id="CHK1", activity="B", event_time=datetime(2024, 1, 1, 11, 0)),
    ]
)
out_df = activity_sequence_number(
    test_df,
    case_col="case_id",
    activity_col="activity",
    timestamp_col="event_time",
    output_col="activity_index",
)
out_df.orderBy("event_time").show(truncate=False)